<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "090"
iso_country = "Solomon Islands"
year = 2018
import_portfolio = "h4_ag6_by_sea_M*.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h4_ag6_by_sea_DX*.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 1.615*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Solomon Islands': 

Number of trade records present in dataset: 12228
Number of partner countries present in dataset: 92
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$478.9m
Total weight within the dataset: 439.7 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,090,Solomon Isds,016,American Samoa,Import,Sea,391910,"391910 | Plastics; plates, sh",3919,3919 | Self-adhesive plates,39,39 | Plastics and article,3692,"Self-adhesive plates, sheets, film, foil, tape...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",4201.232464,NaN,4201.232464,kg,514.124128,514.124128,621.362281,3579.0,2223.886809,1381.839381,['Container'],['Container'],6.632977,6.620723,6.621955,6.621955,6.627431,6.627431,6.943144,12.364097,7.615939,7.116702,7.136788,7.306814,6.796426,12.084262,8.042004,7.382550,7.20844,7.541536,6.667748,13.217883,9.991991,8.744399,8.082705,7.863531,6.541453,14.030137,11.537866,10.761827,8.648080,8.431717,6.417542,14.273142,13.692421,13.843956,9.332101,8.858903,6.291148,13.975639,13.308681,14.540538,10.914591,9.022954,6.172075,12.405675,12.979890,13.583477,10.052441,9.129699,6.050659,11.401587,11.570533,14.133164,11.741103,9.1920,3692
1,1,090,Solomon Isds,016,American Samoa,Import,Sea,400912,400912 | Rubber; vulcanised (,4009,"4009 | Tubes, pipes and hos",40,40 | Rubber and articles,3623,"Tubes, pipes and hoses of vulcanised rubber ot...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",5556.868593,NaN,5556.868593,kg,273.503590,273.503590,576.247273,3579.0,2223.886809,1281.508709,['Container'],['Container'],6.151379,6.140015,6.141157,6.141157,6.146236,6.146236,6.439026,11.466382,7.062971,6.599983,6.618610,6.776291,6.302961,11.206864,7.458101,6.846529,6.68506,6.993970,6.183625,12.258177,9.266506,8.109498,7.495847,7.292586,6.066500,13.011457,10.700140,9.980448,8.020172,7.819519,5.951586,13.236818,12.698261,12.838793,8.654528,8.215688,5.834369,12.960915,12.342383,13.484799,10.122120,8.367828,5.723942,11.504941,12.037464,12.597226,9.322567,8.466822,5.611341,10.573757,10.730436,13.107003,10.888621,8.5246,3623


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $109.1m (22.8%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
10 | Cereals               45525315.0  90408064.1      9.5     20.6
19 | Preparations of cere  15200344.3   7131228.7      3.2      1.6
02 | Meat and edible meat  11107842.6   5373674.0      2.3      1.2
22 | Beverages, spirits a   6126144.2  13384325.1      1.3      3.0
16 | Meat, fish, crustace   5508091.7   1196136.7      1.2      0.3
09 | Coffee, tea, mate an   4245960.1   1181518.7      0.9      0.3
04 | Dairy produce; birds   3489726.4   1583178.3      0.7      0.4
21 | Miscellaneous edible   3146442.8    986507.7      0.7      0.2
15 | Animal, vegetable or   3079388.0   2781782.5      0.6      0.6
20 | Preparations of vege   3050507.0   1703061.5      0.6      0.4
17 | Sugars and sugar con   2599250.2   2812303.6      0.5      0.6
23 | Food indus

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $99.4m (20.7%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Agriculture       59886529.6  81320075.8     12.5     18.5
Food & Beverages  39409296.8  28633448.8      8.2      6.5
Fishing              77649.6     18872.7      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $92.0m (19.2%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
PCR - Processed rice           37039987.8  64338458.9      7.7     14.6
OFD - Food products n.e.c.     29904451.5  13246203.3      6.2      3.0
OMT - Meat products n.e.c.      8995208.8   3702888.0      1.9      0.8
MIL - Dairy products            3657539.0   1588821.4      0.8      0.4
OCR - Crops n.e.c.              3365971.0    876823.1      0.7      0.2
PDR - Paddy rice                2533117.5   4400025.1      0.5      1.0
VOL - Vegetable oils and fats   2208797.7   1995191.5      0.5      0.5
CMT - Bovine meat prods         1891778.5    386708.0      0.4      0.1
V_F - Vegetables, fruit, nuts   1249920.6   1587348.9      0.3      0.4
SGR - Sugar                      986986.8   2250539.2      0.2      0.5
GRO - Cereal grains n.e.c.       159723.2    652767.1      0.0      0.1

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 22.8% 
GTAP 19.2% 
EORA 20.7%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_27852/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,090,Solomon Isds,016,American Samoa,Import,Sea,391910,"391910 | Plastics; plates, sh",3919,3919 | Self-adhesive plates,39,39 | Plastics and article,3692,"Self-adhesive plates, sheets, film, foil, tape...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",4201.232464,NaN,4201.232464,kg,514.124128,514.124128,621.362281,3579.0,2223.886809,1381.839381,['Container'],['Container'],6.632977,6.620723,6.621955,6.621955,6.627431,6.627431,6.943144,12.364097,7.615939,7.116702,7.136788,7.306814,6.796426,12.084262,8.042004,7.382550,7.20844,7.541536,6.667748,13.217883,9.991991,8.744399,8.082705,7.863531,6.541453,14.030137,11.537866,10.761827,8.648080,8.431717,6.417542,14.273142,13.692421,13.843956,9.332101,8.858903,6.291148,13.975639,13.308681,14.540538,10.914591,9.022954,6.172075,12.405675,12.979890,13.583477,10.052441,9.129699,6.050659,11.401587,11.570533,14.133164,11.741103,9.1920,3692,NaN
1,1,090,Solomon Isds,016,American Samoa,Import,Sea,400912,400912 | Rubber; vulcanised (,4009,"4009 | Tubes, pipes and hos",40,40 | Rubber and articles,3623,"Tubes, pipes and hoses of vulcanised rubber ot...","Petroleum, Chemical and Non-Metallic Mineral P...",33,"CRP - Chemical, rubber, plastic products",5556.868593,NaN,5556.868593,kg,273.503590,273.503590,576.247273,3579.0,2223.886809,1281.508709,['Container'],['Container'],6.151379,6.140015,6.141157,6.141157,6.146236,6.146236,6.439026,11.466382,7.062971,6.599983,6.618610,6.776291,6.302961,11.206864,7.458101,6.846529,6.68506,6.993970,6.183625,12.258177,9.266506,8.109498,7.495847,7.292586,6.066500,13.011457,10.700140,9.980448,8.020172,7.819519,5.951586,13.236818,12.698261,12.838793,8.654528,8.215688,5.834369,12.960915,12.342383,13.484799,10.122120,8.367828,5.723942,11.504941,12.037464,12.597226,9.322567,8.466822,5.611341,10.573757,10.730436,13.107003,10.888621,8.5246,3623,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $85.7m of $478.9m (17.9%)
Volume: 114.6 of 439.7 million kilograms (26.1%)
Value-to-Weight: 0.75 from 1.09 US$ per kg (+-31.3%)
Average distance to market (volume-weighted): 3179.3 from 3400.2 nm (+-6.5%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    321  59374115.5  78771738.6     12.4     17.9
1        Meat and Prep     44   9332406.8   5001495.9      1.9      1.1
2  Unprocessed Cereals     18   5952209.6  21669580.2      1.2      4.9
3                Dairy     53   3175869.6   1407248.8      0.7      0.3
4      Sugar and Honey     57   2605900.0   2814210.1      0.5      0.6
5        Fats and Oils     65   2581881.5   2253637.7      0.5      0.5
6           Vegetables    109   2114552.6   2186250.4      0.4      0.5
7               Fruits     44    538413.4    474096.7      0.1      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,1671509.7,2975218.4,1833238.8,1713129.1,1717958.2,1758945.9,1545107.6,3434442.9,3294877.6,3331679.1,2246112.4,2132175.8,1456875.8,2743973.8,2784555.6,3400498.3,2825182.0,2212270.3
1,Fats and Oils,3521662.6,5793500.0,3778488.4,3553750.5,3559669.4,3665291.4,3302916.1,6591426.1,6381479.8,6583214.3,4521061.4,4300921.6,3150225.5,5457417.6,5508256.6,6459725.4,5448457.3,4435900.0
2,Fruits,603414.1,1071984.5,661380.7,618302.5,620017.8,634820.8,557975.7,1237832.7,1186931.0,1200551.8,810045.3,768978.7,526258.6,990166.1,1004686.4,1225208.6,1017439.3,797786.4
3,Meat and Prep,6533647.4,11634880.7,7166762.0,6696969.6,6715870.7,6875868.5,6039045.2,13431333.5,12884862.2,13027460.0,8781707.2,8336417.9,5693800.2,10729138.8,10888120.0,13299610.9,11048630.2,8649869.5
4,Processed Cereals,127182718.0,223351076.1,138895249.5,130011873.7,130374499.2,133591794.6,117855631.9,258905626.0,248091789.6,251506266.1,170255416.5,161044893.7,111345102.3,207263660.3,210332488.9,255180284.6,212421012.3,166905669.3
5,Sugar and Honey,4024632.5,6596817.9,4303688.2,4061694.7,4072025.4,4189332.8,3774850.9,7792835.0,7428538.8,7634803.8,5275980.4,4910953.4,3600503.3,6316209.0,6407443.9,7508972.4,6314721.1,5060519.9
6,Unprocessed Cereals,30870588.3,54505420.5,33770518.0,31589695.4,31678424.8,32449839.1,28578528.0,63087242.0,60478812.5,61246871.6,41393440.4,39204675.2,26978611.3,50457335.0,51205298.0,62287515.9,51811206.1,40649545.5
7,Vegetables,3217528.4,5462360.4,3477672.0,3270058.4,3277289.1,3364232.5,3000125.0,6348684.8,6064983.8,6198218.8,4237023.9,4000044.9,2848335.4,5141758.7,5212997.0,6213366.3,5185750.6,4135121.0


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.41,0.05,0.01,0.01,0.03,0.59,0.55,0.56,0.22,0.18,0.41,0.42,0.61,0.43,0.24
1,Fats and Oils,0.88,0.10,0.01,0.01,0.06,1.27,1.19,1.27,0.47,0.39,0.89,0.91,1.28,0.89,0.50
2,Fruits,0.87,0.11,0.03,0.03,0.06,1.26,1.17,1.19,0.47,0.39,0.86,0.89,1.30,0.91,0.50
3,Meat and Prep,0.55,0.07,0.02,0.02,0.04,0.79,0.73,0.75,0.29,0.25,0.54,0.56,0.81,0.57,0.32
4,Processed Cereals,1.62,0.20,0.05,0.05,0.11,2.38,2.19,2.25,0.88,0.73,1.62,1.67,2.42,1.70,0.94
5,Sugar and Honey,0.99,0.11,0.01,0.02,0.06,1.54,1.40,1.48,0.58,0.44,1.04,1.08,1.50,1.04,0.56
6,Unprocessed Cereals,3.97,0.49,0.12,0.14,0.27,5.80,5.36,5.49,2.15,1.79,3.94,4.07,5.93,4.17,2.30
7,Vegetables,1.06,0.12,0.02,0.03,0.07,1.58,1.45,1.51,0.58,0.47,1.08,1.12,1.59,1.11,0.61
